# [Tutorial](https://github.com/biohub/esm/tree/main/cookbook/tutorials): ESMFold2

## **Overview**

ESMFold2 is a structure prediction model that can fold proteins, nucleic acids (RNA and DNA), small-molecule ligands, and complexes containing combinations of these. In this tutorial, we use the `fold_all_atom()` method to predict all-atom structures for three systems that highlight different capabilities:

1. **Protein–nucleic acid complexes**:

    LC11-RNase H1 (a thermostable metagenomic RNase H1) bound to an RNA:DNA hybrid duplex. This example shows how to fold a protein together  with RNA and DNA chains.

2. **Non-canonical amino acids + Covalent modifications**:

    The GLP-1 receptor (GLP1R) bound to semaglutide, a therapeutic peptide with a non-canonical amino acid and a covalently attached lipid linker. This example shows how to include non-canonical residues via [Chemical Component Dictionary (CCD) codes](https://www.wwpdb.org/data/ccd), define a small-molecule ligand from SMILES, and specify an explicit covalent bond between a peptide and a ligand.

3. **Folding with MSAs**:

    We use the example of Ubiquitin to show how to incorporate evolutionary information to improve prediction accuracy.

## **Learning goals**

After completing this tutorial, you will be able to:

- Use the ESM SDK `input_builder` to represent multi-chain biological systems
- Fold protein–RNA–DNA complexes with `fold_all_atom()`
- Include non-canonical amino acids in a peptide via CCD codes
- Define small-molecule ligands from SMILES
- Specify explicit covalent bonds between peptide and ligand atoms
- Pass a multiple sequence alignment (MSA) to improve folding accuracy
- Interpret per-residue (pLDDT) and pairwise (PAE) confidence outputs
- Apply this workflow to your own systems


**Note that each example can be run independently after completing the Setup section**

# **Setup**

In [2]:
# If you are working in colab, uncomment these lines to install dependencies
!pip install esm --ignore-requires-python
!pip install py3dmol

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.8/184.8 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2

In [3]:
from getpass import getpass

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol

from esm.sdk import esmfold2_client
from esm.sdk.api import FoldingConfig
from esm.utils.structure import input_builder

In [4]:
from IPython.display import HTML, display


def plddt_hex(v):
    """Convert pLDDT score to hex color."""
    if v >= 90:
        return "#0053D6"
    if v >= 70:
        return "#65CBF3"
    if v >= 50:
        return "#FFDB13"
    return "#FF7D45"


PLDDT_LEGEND = (
    '<span style="color:#FF7D45;">&#9632;</span> &lt;50 &nbsp;'
    '<span style="color:#FFDB13;">&#9632;</span> 50–70 &nbsp;'
    '<span style="color:#65CBF3;">&#9632;</span> 70–90 &nbsp;'
    '<span style="color:#0053D6;">&#9632;</span> &gt;90'
)


def normalize_plddt(plddt):
    plddt = np.array(plddt)
    return plddt * 100 if plddt.max() <= 1.0 else plddt


def apply_plddt_style(view, plddt, add_hetflag=False):
    view.setStyle({}, {})
    for i, s in enumerate(plddt):
        view.setStyle({"resi": i + 1}, {"cartoon": {"color": plddt_hex(s)}})
    if add_hetflag:
        view.addStyle({"hetflag": True}, {"stick": {}})
    view.zoomTo()


def show_dual_viewer(v1, title1, legend1, v2, title2, legend2):
    l1 = (
        f'<p style="text-align:center; font-size:12px; margin-top:4px;">{legend1}</p>'
        if legend1
        else ""
    )
    l2 = (
        f'<p style="text-align:center; font-size:12px; margin-top:4px;">{legend2}</p>'
        if legend2
        else ""
    )
    display(
        HTML(
            '<div style="display:flex; gap:20px;">'
            f'<div><p style="text-align:center; font-weight:bold;">{title1}</p>'
            + v1._make_html()
            + l1
            + "</div>"
            f'<div><p style="text-align:center; font-weight:bold;">{title2}</p>'
            + v2._make_html()
            + l2
            + "</div>"
            "</div>"
        )
    )


def plot_pae(pae, chains=None, title=None):
    """Plot a PAE matrix with optional per-chain colored bars and legend.

    Args:
        pae: PAE as a numpy array (n x n).
        chains: list of (start, end, label, color) tuples. When provided, colored
            bars are drawn along the top and left axes and a legend is shown.
        title: optional plot title.
    """
    total_len = pae.shape[0]

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(
        pae, cmap="Greens_r", vmin=0, vmax=30, origin="upper", aspect="equal"
    )

    if chains:
        bar_thickness = total_len * 0.015
        for start, end, label, color in chains:
            ax.add_patch(
                mpatches.Rectangle(
                    (start - 0.5, -bar_thickness - 1),
                    end - start,
                    bar_thickness,
                    facecolor=color,
                    clip_on=False,
                    edgecolor="black",
                    linewidth=0.5,
                )
            )
            ax.add_patch(
                mpatches.Rectangle(
                    (-bar_thickness - 1, start - 0.5),
                    bar_thickness,
                    end - start,
                    facecolor=color,
                    clip_on=False,
                    edgecolor="black",
                    linewidth=0.5,
                )
            )
        for start, *_ in chains[1:]:
            ax.axhline(start - 0.5, color="black", linewidth=1.5)
            ax.axvline(start - 0.5, color="black", linewidth=1.5)
        legend_handles = [
            mpatches.Patch(facecolor=c, edgecolor="black", label=l)
            for _, _, l, c in chains
        ]
        ax.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.08),
            ncol=len(chains),
            frameon=True,
            fontsize=10,
            title="Chain",
        )
        ax.set_xlim(-bar_thickness - 2, total_len)
        ax.set_ylim(total_len, -bar_thickness - 2)

    ax.set_xlabel("Scored Residue", fontsize=12)
    ax.set_ylabel("Aligned Residue", fontsize=12)
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", pad=15)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Expected Position Error (Ångströms)", fontsize=12)
    cbar.set_ticks([0, 5, 10, 15, 20, 25, 30])
    cbar.ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

## Biohub Client

First, we need to set up the Biohub client. Generate an API key on the [Biohub platform](https://biohub.ai) and add it to your account. This API key manages your access to credits and tokens, and the term API key/token is often used interchangeably within documentation.

Please note that your API key is like a password for your account and you should take care to protect it. For this reason it is recommended to frequently create a new API key and delete old, unused ones. It is also recommended to paste the API key directly into an environment variable or use a utility like `getpass` as shown in the example below so keys are not accidentally shared or checked into code repositories.

In [5]:
token = getpass("Biohub token: ")

client = esmfold2_client(model="esmfold2-2026-05", url="https://biohub.ai", token=token)

Biohub token: ··········


# **Folding Configuration**

Configure the folding parameters to control prediction quality and computational cost:

* **num_loops** (default `20`): Number of times the structure is iteratively refined. More loops generally improve accuracy but increase runtime.

* **num_sampling_steps** (default `100`): Number of diffusion sampling steps. Higher values produce more accurate structures but take longer.

* **lm_dropout** (default `0.3`): Dropout probability on LM pair embeddings. When > 0, dropout is applied.

* **lm_mask_pct** (default `0.1` for `esmfold2-fast`, `0.0` for `esmfold2`): Fraction of sequence residues randomly masked before the PLM backbone. Leave unset (`None`) to use the model-dependent default.

* **msa_max_depth** (default `1024`): Number of MSA rows randomly subsampled on each loop. Set to `None` to disable MSA subsampling.

We use a moderate number of loops (10) to balance accuracy and runtime for this example. For more accurate results, especially when modeling flexible ligands or uncertain binding modes, you can increase num_loops at the cost of longer runtimes.

In [11]:
config = FoldingConfig(num_loops=10, num_sampling_steps=100, include_pae=True)

# **Example 1: RNase H1–RNA/DNA hybrid complex**
Ribonuclease H1 (RNase H1) is an endonuclease that specifically hydrolyzes the RNA strand of an RNA:DNA hybrid. RNase H activity is essential during DNA replication, transcription, and reverse transcription, and is the mechanism of action exploited by gapmer antisense oligonucleotide drugs.

The complex from PDB 4H8K has three biological roles distributed across four chains:

- **LC11-RNase H1 (chains A and B):** a thermostable RNase H1 ortholog from a metagenomic source, expressed and crystallized as a homodimer. We supply it as a single `ProteinInput` with two chain IDs.
- **Substrate RNA (chain C):** the RNA strand that pairs with the DNA template and is cleaved by the enzyme.
- **Substrate DNA (chain D):** the complementary DNA strand of the hybrid duplex.

We will fold all four chains together, replicating the architecture seen in PDB 4H8K.

In [12]:
# target_protein = ("RSQLDRVEALLSPEDKISLNMAKPFRELEPELVTRRKNDFQRLYTNDREDYLGKLERDITKFFVDRGFLEIKSPILIPAEYVERMGITPETPLAKQIFRVDKNLCLRPTPAPAIYDEIRRLDRILPGPIRLFTVGPGYRKESDGKEHLEEFTSATFAQMGSGCTRENLEALIKEFLDYLEIDFEIVGDDHPLAGDTLDIMHGDLELGIAFVGPVSLDREWGIDKPGISAGFGLERLLKVMHGFKNIKRASRSESYYNGISTNL")
# receptor = input_builder.ProteinInput(id="A", sequence=target_protein)
# def get_protractor_smiles() -> str:
#     return "CC(=O)NCCCC[C@@H](C(=O)O)N"


# ligand = input_builder.LigandInput(id="C", ccd=None, smiles=get_protractor_smiles())
# complex_inputs = input_builder.StructurePredictionInput(
#     sequences=[receptor, ligand])
# complex_structure = client.fold_all_atom(complex_inputs, config=config)

# with open("Hit1_AcKRS3.cif", "w") as f:
#     f.write(complex_structure.complex.to_mmcif())

target_protein = "RSQLDRVEALLSPEDKISLNMAKPFRELEPELVTRRKNDFQRLYTNDREDYLGKLERDITKFFVDRGFLEIKSPILIPAEYVERMGITPETPLAKQIFRVDKNLCLRPTPAPAIYDEIRRLDRILPGPIRLFTVGPGYRKESDGKEHLEEFTSATFAQMGSGCTRENLEALIKEFLDYLEIDFEIVGDDHPLAGDTLDIMHGDLELGIAFVGPVSLDREWGIDKPGISAGFGLERLLKVMHGFKNIKRASRSESYYNGISTNL"

# Define the receptor protein
receptor = input_builder.ProteinInput(id="A", sequence=target_protein)

# Define the free-floating ligand (Omit ccd=None entirely)
protractor_smiles = "CC(=O)NCCCC[C@@H](C(=O)O)N"
ligand = input_builder.LigandInput(id="C", smiles=protractor_smiles)

# Build the structural prediction input container
complex_inputs = input_builder.StructurePredictionInput(
    sequences=[receptor, ligand]
)

# Call the model
complex_structure = client.fold_all_atom(complex_inputs, config=config)

# Gracefully handle the response object type before checking attributes
if hasattr(complex_structure, 'error') or "Error" in type(complex_structure).__name__:
    print(f"Prediction failed with server message: {complex_structure}")
else:
    # Execution was successful, write out the mmCIF file structure
    with open("Hit1_AcKRS3.cif", "w") as f:
        f.write(complex_structure.complex.to_mmcif())
    print("Structure successfully written to Hit1_AcKRS3.cif!")



Structure successfully written to Hit1_AcKRS3.cif!


In [13]:
cif_data = open("Hit1_AcKRS3.cif").read()
plddt = normalize_plddt(complex_structure.plddt.cpu().numpy())

v1 = py3Dmol.view(width=500, height=400)
v1.addModel(cif_data, "mmcif")
v1.setStyle({"chain": "A"}, {"cartoon": {"color": "#2166ac"}})
v1.addStyle({"hetflag": True}, {"stick": {"color": "#e6550d"}})
v1.zoomTo()

v2 = py3Dmol.view(width=500, height=400)
v2.addModel(cif_data, "mmcif")
apply_plddt_style(v2, plddt, add_hetflag=True)
chain_legend = (
    '<span style="color:#2166ac;">&#9632;</span> Receptor &nbsp;'
    '<span style="color:#f4a9c4;">&#9632;</span> Peptide &nbsp;'
    '<span style="color:#e6550d;">&#9632;</span> Protractor'
)
show_dual_viewer(
    v1, "Chain identity", chain_legend, v2, "Confidence (pLDDT)", PLDDT_LEGEND
)